In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt

import joblib
import json
import os
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# Create folders
# ==========================================

os.makedirs('../models', exist_ok=True)
os.makedirs('../plots', exist_ok=True)

# ==========================================
# Load Dataset
# ==========================================

df = pd.read_csv('../data/cc_segmented.csv')

print("Dataset Loaded")
print(df.shape)

# ==========================================
# Features for Anomaly Detection
# ==========================================

ANOMALY_FEATURES = [

    'monthly_spend',

    'txn_count',

    'avg_txn_value',

    'util_rate',

    'intl_txn_pct',

    'travel_spend',

    'reward_pts_mo'
]

X = df[ANOMALY_FEATURES].fillna(0)

# ==========================================
# Scaling
# ==========================================

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

# ==========================================
# Isolation Forest Model
# ==========================================

print()
print("Training Isolation Forest...")

iso_model = IsolationForest(

    n_estimators=120,

    contamination=0.04,

    random_state=42,

    n_jobs=-1
)

iso_model.fit(X_scaled)

# ==========================================
# Predictions
# ==========================================

df['anomaly_score'] = (
    -iso_model.score_samples(X_scaled)
)

df['is_anomaly'] = (
    iso_model.predict(X_scaled) == -1
).astype(int)

# ==========================================
# Metrics
# ==========================================

n_anomalies = df['is_anomaly'].sum()

anomaly_rate = (
    n_anomalies / len(df)
) * 100

print()
print("ANOMALY DETECTION RESULTS")
print("-" * 40)

print(f"Anomalies Found : {n_anomalies:,}")

print(f"Anomaly Rate    : {anomaly_rate:.2f}%")

# ==========================================
# Compare Normal vs Anomaly
# ==========================================

print()
print("ANOMALY vs NORMAL")
print("-" * 40)

for feat in [

    'monthly_spend',

    'util_rate',

    'txn_count',

    'intl_txn_pct'
]:

    anom_mean = df[
        df['is_anomaly'] == 1
    ][feat].mean()

    norm_mean = df[
        df['is_anomaly'] == 0
    ][feat].mean()

    print(
        f"{feat:20} | "
        f"Anomaly={anom_mean:.2f} | "
        f"Normal={norm_mean:.2f}"
    )

# ==========================================
# Visualization
# ==========================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12,5)
)

# ==========================================
# Scatter Plot
# ==========================================

normal = df[
    df['is_anomaly'] == 0
].sample(1500)

anomaly = df[
    df['is_anomaly'] == 1
]

axes[0].scatter(

    normal['monthly_spend']/1000,

    normal['util_rate'],

    alpha=0.2,

    s=10,

    color='#2563EB',

    label='Normal'
)

axes[0].scatter(

    anomaly['monthly_spend']/1000,

    anomaly['util_rate'],

    alpha=0.7,

    s=25,

    color='red',

    label='Anomaly'
)

axes[0].set_title(
    'Spend vs Utilization',
    fontweight='bold'
)

axes[0].set_xlabel(
    'Monthly Spend (Thousands)'
)

axes[0].set_ylabel(
    'Utilization Rate'
)

axes[0].legend()

# ==========================================
# Anomaly Score Distribution
# ==========================================

axes[1].hist(

    df[df['is_anomaly']==0]['anomaly_score'],

    bins=40,

    alpha=0.6,

    density=True,

    label='Normal'
)

axes[1].hist(

    df[df['is_anomaly']==1]['anomaly_score'],

    bins=40,

    alpha=0.7,

    density=True,

    label='Anomaly'
)

axes[1].set_title(
    'Anomaly Score Distribution',
    fontweight='bold'
)

axes[1].legend()

plt.tight_layout()

plt.savefig(

    '../plots/anomaly_analysis.png',

    dpi=100,

    bbox_inches='tight'
)

plt.close()

# ==========================================
# Save Models
# ==========================================

joblib.dump(

    iso_model,

    '../models/anomaly_detector.pkl'
)

joblib.dump(

    scaler,

    '../models/anomaly_scaler.pkl'
)

# ==========================================
# Save Metrics
# ==========================================

json.dump(

    {

        'anomaly_features':
            ANOMALY_FEATURES,

        'contamination': 0.04,

        'n_anomalies':
            int(n_anomalies),

        'anomaly_rate':
            round(anomaly_rate, 2)
    },

    open('../models/anomaly_metrics.json', 'w'),

    indent=2
)

# ==========================================
# Save Dataset
# ==========================================

df.to_csv(

    '../data/cc_final.csv',

    index=False
)

print()
print("Saved:")
print("models/anomaly_detector.pkl")
print("models/anomaly_scaler.pkl")
print("models/anomaly_metrics.json")
print("plots/anomaly_analysis.png")
print("data/cc_final.csv")

Dataset Loaded
(30000, 24)

Training Isolation Forest...

ANOMALY DETECTION RESULTS
----------------------------------------
Anomalies Found : 1,200
Anomaly Rate    : 4.00%

ANOMALY vs NORMAL
----------------------------------------
monthly_spend        | Anomaly=191781.04 | Normal=50937.06
util_rate            | Anomaly=0.71 | Normal=0.52
txn_count            | Anomaly=15.64 | Normal=16.04
intl_txn_pct         | Anomaly=0.11 | Normal=0.06

Saved:
models/anomaly_detector.pkl
models/anomaly_scaler.pkl
models/anomaly_metrics.json
plots/anomaly_analysis.png
data/cc_final.csv
